In [ ]:
!pip install uv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 153.3 MB/s eta 0:00:00


In [ ]:
# 1. Install required downloaders
!uv pip install -q kaggle ipykernel jupyter_client huggingface_hub

In [ ]:
# 2. Upload kaggle.json for authentication
print(">>> Please click 'Choose Files' below and upload your kaggle.json file:")
from google.colab import files
import os

files.upload()

# (Optional but helpful) ensure kaggle CLI exists
!pip -q install kaggle huggingface_hub

# 3. Move kaggle.json to the correct hidden folder
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# 4. Download the custom Python wheels/tools from Kaggle
print("\n>>> Downloading custom Python tools from Kaggle...")
!mkdir -p /content/aimo_utils
!kaggle kernels output andreasbis/aimo-3-utils -p /content/aimo_utils

# 5. Download the 120B Model from Hugging Face (excluding metal/ and original/)
print("\n>>> Downloading 120B Model weights (excluding metal/ and original/)...")
from huggingface_hub import snapshot_download

out_dir = "/content/model_weights/transformers/default/1"
os.makedirs(out_dir, exist_ok=True)

snapshot_download(
    repo_id="openai/gpt-oss-120b",
    local_dir=out_dir,
    # keep it similar to your original cell: download these file types
    allow_patterns=["*.safetensors", "*.json", "*.jinja", "*.md"],
    # exclude variant folders
    ignore_patterns=["metal/**", "original/**"],
)

print("\n>>> Downloading Eagle3 Draft Model weights...")
draft_dir = "/content/model_weights/draft"
os.makedirs(draft_dir, exist_ok=True)
snapshot_download(
    repo_id="wenliang1990/gpt-oss-120b-eagle3-aimo3",
    local_dir=draft_dir,
    allow_patterns=["*.safetensors", "*.json", "*.jinja", "*.md"],
)


print("\n>>> ALL DOWNLOADS COMPLETE! Move to the next cell.")

>>> Please click 'Choose Files' below and upload your kaggle.json file:


Saving kaggle.json to kaggle.json

>>> Downloading custom Python tools from Kaggle...
Output file downloaded to /content/aimo_utils/wheels.tar.gz
Kernel log downloaded to /content/aimo_utils/aimo-3-utils.log 

>>> Downloading 120B Model weights (excluding metal/ and original/)...


Fetching 23 files:   0%|          | 0/23 [00:00<?, ?it/s]


>>> Downloading Eagle3 Draft Model weights...


Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]


>>> ALL DOWNLOADS COMPLETE! Move to the next cell.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%pip uninstall --yes 'keras' 'matplotlib' 'scikit-learn' 'tensorflow'

Found existing installation: keras 3.13.2
Uninstalling keras-3.13.2:
  Successfully uninstalled keras-3.13.2
Found existing installation: matplotlib 3.10.0
Uninstalling matplotlib-3.10.0:
  Successfully uninstalled matplotlib-3.10.0
Found existing installation: scikit-learn 1.6.1
Uninstalling scikit-learn-1.6.1:
  Successfully uninstalled scikit-learn-1.6.1
Found existing installation: tensorflow 2.19.0
Uninstalling tensorflow-2.19.0:
  Successfully uninstalled tensorflow-2.19.0


In [ ]:
import warnings
warnings.simplefilter('ignore')

In [ ]:
import os
import sys
import subprocess

In [ ]:
def set_env(input_archive, temp_dir):

    if not os.path.exists(temp_dir):
        os.makedirs(temp_dir, exist_ok=True)

        subprocess.run(['tar', '-xzf', input_archive, '-C', temp_dir], check=True)

    subprocess.run([
        sys.executable,
        '-m',
        'pip',
        'install',
        '--no-index',
        '--find-links',
        f'{temp_dir}/wheels',
        'unsloth',
        'trl',
        'vllm',
        'openai_harmony'
    ], check=True)

In [ ]:
set_env(
    input_archive='/content/aimo_utils/wheels.tar.gz',
    temp_dir='/content/setup'
)

In [ ]:
subprocess.run(['ls', '/content/setup/tiktoken_encodings'])

CompletedProcess(args=['ls', '/content/setup/tiktoken_encodings'], returncode=0)

In [ ]:
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['TRITON_PTXAS_PATH'] = '/usr/local/cuda/bin/ptxas'
os.environ['TIKTOKEN_ENCODINGS_BASE'] = '/content/setup/tiktoken_encodings'

os.environ['VLLM_ATTENTION_BACKEND'] = 'TRITON_ATTN'


In [ ]:
import gc
import re
import math
import time
import queue
import threading
import contextlib
from typing import Optional
from jupyter_client import KernelManager
from collections import Counter, defaultdict
from concurrent.futures import as_completed, ThreadPoolExecutor

import pandas as pd
import polars as pl

from openai import OpenAI

from openai_harmony import (
    HarmonyEncodingName,
    load_harmony_encoding,
    SystemContent,
    ReasoningEffort,
    ToolNamespaceConfig,
    Author,
    Message,
    Role,
    TextContent,
    Conversation
)

from transformers import set_seed

In [ ]:
class CFG:

    system_prompt = (
        'You are an elite mathematical problem solver with expertise at the International '
        'Mathematical Olympiad (IMO) level. Your goal is to find the correct answer through '
        'rigorous mathematical reasoning.\n\n'

        '# Problem-Solving Approach:\n'
        '1. UNDERSTAND: Carefully read and rephrase the problem in your own words. '
        'Identify what is given, what needs to be found, and any constraints.\n'
        '2. EXPLORE: Consider multiple solution strategies. Think about relevant theorems, '
        'techniques, patterns, or analogous problems. Don\'t commit to one approach immediately.\n'
        '3. PLAN: Select the most promising approach and outline key steps before executing.\n'
        '4. EXECUTE: Work through your solution methodically. Show all reasoning steps clearly.\n'
        '5. VERIFY: Check your answer by substituting back, testing edge cases, or using '
        'alternative methods. Ensure logical consistency throughout.\n\n'

        '# Mathematical Reasoning Principles:\n'
        '- Break complex problems into smaller, manageable sub-problems\n'
        '- Look for patterns, symmetries, and special cases that provide insight\n'
        '- Use concrete examples to build intuition before generalizing\n'
        '- Consider extreme cases and boundary conditions\n'
        '- If stuck, try working backwards from the desired result\n'
        '- Be willing to restart with a different approach if needed\n\n'

        '# Verification Requirements:\n'
        '- Cross-check arithmetic and algebraic manipulations\n'
        '- Verify that your solution satisfies all problem constraints\n'
        '- Test your answer with simple cases or special values when possible\n'
        '- Ensure dimensional consistency and reasonableness of the result\n\n'

        '# Output Format:\n'
        'The final answer must be a non-negative integer between 0 and 99999.\n'
        'Place your final numerical answer inside \\boxed{}, e.g., \\boxed{42}\n\n'

        'Think step-by-step and show your complete reasoning process. Quality of reasoning '
        'is as important as the final answer.'
    )

    tool_prompt = (
        'Use this tool to execute Python code for:\n'
        '- Complex calculations that would be error-prone by hand\n'
        '- Numerical verification of analytical results\n'
        '- Generating examples or testing conjectures\n'
        '- Visualizing problem structure when helpful\n'
        '- Brute-force verification for small cases\n\n'

        'The environment is a stateful Jupyter notebook. Code persists between executions.\n'
        'Always use print() to display results. Write clear, well-commented code.\n\n'

        'Remember: Code should support your mathematical reasoning, not replace it. '
        'Explain what you\'re computing and why before running code.'
    )

    preference_prompt = (
        'You have access to `math`, `numpy`, `scipy`, and `sympy` for:\n\n'

        '# Symbolic Computation (sympy):\n'
        '- Algebraic manipulation and simplification\n'
        '- Solving equations and systems of equations\n'
        '- Symbolic differentiation and integration\n'
        '- Number theory functions (primes, divisors, modular arithmetic)\n'
        '- Polynomial operations and factorization\n'
        '- Working with mathematical expressions symbolically\n\n'

        '# Advanced Mathematics (scipy):\n'
        '- Combinatorics (e.g., scipy.special.comb)\n'
        '- Special mathematical functions\n'
        '- Optimization and root-finding\n\n'

        '# Numerical Computation (numpy):\n'
        '- Array operations and linear algebra\n'
        '- Efficient numerical calculations for large datasets\n'
        '- Matrix operations and eigenvalue problems\n'
        '- Statistical computations\n\n'

        '# Mathematical Functions (math):\n'
        '- Standard mathematical functions (trig, log, exp)\n'
        '- Constants like pi and e\n'
        '- Basic operations for single values\n\n'

        'Best Practices:\n'
        '- Use sympy for exact symbolic answers when possible\n'
        '- Use numpy for numerical verification and large-scale computation\n'
        '- Combine symbolic and numerical approaches: derive symbolically, verify numerically\n'
        '- Document your computational strategy clearly\n'
        '- Validate computational results against known cases or theoretical bounds'
    )

    summarizer_system_prompt = (
        "You are an expert mathematical technical writer. Your job is to read a raw "
        "problem-solving trace produced by a code agent and provide a summary. "
        "Do not attempt to write code or re-solve the problem."
        "Your role is to objectively summarize the solution presented in the shared trace."

        "It is very important not to whitewash the solution, since it will be compared "
        "against different proposed solutions. "

        "Summarize how the agent understood the question, the agent’s key hypotheses, "
        "the exact mathematical reasoning, and the core logic from the provided trace. "


        "You may include notes about the agent’s understanding of the problem, its key "
        "assumptions, boundaries of the solution, inclusions, exclusions, and related "
        "details. Be neutral, but make sure to capture the important details of the "
        "solution. Your summary must be fully auditable by the judge, without any confusion. "
        "Do not forget to include code blocks where relevant."
    )

    summarizer_reasoning_effort = ReasoningEffort.HIGH

    served_model_name = 'gpt-oss'
    model_path = '/content/model_weights/transformers/default/1' # Keep Colab path!
    draft_model_path = '/content/model_weights/draft'


    kv_cache_dtype = 'fp8_e4m3'
    dtype = 'auto'

    # --- EXACT KAGGLE PARAMETERS ---
    high_problem_timeout = 1500
    base_problem_timeout = 0

    max_judger_calls = 50      # Disables Judger, just like Kaggle
    judger_timeout = 500

    notebook_limit = 60000
    server_timeout = 180

    session_timeout = 960
    jupyter_timeout = 20
    sandbox_timeout = 3

    stream_interval = 200
    context_tokens = 65536
    buffer_tokens = 512
    search_tokens = 32
    batch_size = 32

    early_stop_unanimous = 3
    early_stop = 3
    attempts = 5
    judger_attempts = 3
    workers = 9
    turns = 128
    seed = 42

    gpu_memory_utilization = 0.96
    temperature = 1.0

    save_traces = True
    trace_dir = '/content/traces'
    verbose_traces = False

In [ ]:
set_seed(CFG.seed)

In [ ]:
class AIMO3Template:
    def get_system_content(self, system_prompt: str, tool_config: ToolNamespaceConfig) -> SystemContent:
        return SystemContent.new().with_model_identity(system_prompt).with_reasoning_effort(ReasoningEffort.HIGH).with_tools(tool_config)

    def apply_chat_template(self, system_prompt: str, user_prompt: str, tool_config: ToolNamespaceConfig) -> list[Message]:
        return [
            Message.from_role_and_content(Role.SYSTEM, self.get_system_content(system_prompt, tool_config)),
            Message.from_role_and_content(Role.USER, user_prompt)
        ]


In [ ]:
class AIMO3Sandbox:
    sys.set_int_max_str_digits(0)
    _port_lock = threading.Lock()
    _next_port = 50000

    @classmethod
    def _get_next_ports(cls, count: int = 5) -> list[int]:
        with cls._port_lock:
            ports = list(range(cls._next_port, cls._next_port + count))
            cls._next_port += count
            return ports

    def __init__(self, timeout: float):
        self._default_timeout = timeout
        self._owns_kernel = False
        ports = self._get_next_ports(5)

        env = os.environ.copy()
        env['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'
        env['PYDEVD_WARN_EVALUATION_TIMEOUT'] = '0'
        env['JUPYTER_PLATFORM_DIRS'] = '1'
        env['PYTHONWARNINGS'] = 'ignore'
        env['MPLBACKEND'] = 'Agg'

        self._km = KernelManager()
        self._km.shell_port, self._km.iopub_port, self._km.stdin_port, self._km.hb_port, self._km.control_port = ports
        self._km.start_kernel(env=env, extra_arguments=['--Application.log_level=CRITICAL'])

        self._client = self._km.blocking_client()
        self._client.start_channels()
        self._client.wait_for_ready(timeout=self._default_timeout)
        self._owns_kernel = True

        # Updated to initialize all required math libraries
        self.execute(
            'import sys\nsys.set_int_max_str_digits(0)\nsys.setrecursionlimit(20000)\n'
            'import math\nimport numpy\nimport sympy\nimport scipy\nimport scipy.special\n'
            'import itertools\nimport collections\nimport mpmath\nmpmath.mp.dps = 64\n'
        )

    def _format_error(self, traceback: list[str]) -> str:
        clean_lines = []
        for frame in traceback:
            clean_frame = re.sub(r'\x1b\[[0-9;]*m', '', frame)
            if 'File "' in clean_frame and 'ipython-input' not in clean_frame:
                continue
            clean_lines.append(clean_frame)
        return ''.join(clean_lines)

    def execute(self, code: str, timeout: float | None = None) -> str:
        client = self._client
        effective_timeout = timeout or self._default_timeout
        msg_id = client.execute(code, store_history=True, allow_stdin=False, stop_on_error=False)

        stdout_parts, stderr_parts = [], []
        start_time = time.time()

        while True:
            if time.time() - start_time > effective_timeout:
                self._km.interrupt_kernel()
                return f'[ERROR] Execution timed out after {effective_timeout} seconds'

            try:
                msg = client.get_iopub_msg(timeout=1.0)
            except queue.Empty:
                continue

            if msg.get('parent_header', {}).get('msg_id') != msg_id: continue
            msg_type = msg.get('msg_type')
            content = msg.get('content', {})

            if msg_type == 'stream':
                (stdout_parts if content.get('name') == 'stdout' else stderr_parts).append(content.get('text', ''))
            elif msg_type == 'error':
                # Apply error formatting to strip ANSI codes
                stderr_parts.append(self._format_error(content.get('traceback', [])))
            elif msg_type in {'execute_result', 'display_data'}:
                text = content.get('data', {}).get('text/plain')
                if text: stdout_parts.append(text + '\n')
            elif msg_type == 'status' and content.get('execution_state') == 'idle':
                break

        stdout = ''.join(stdout_parts)
        stderr = ''.join(stderr_parts)
        if stderr: return f'{stdout.rstrip()}\n{stderr}' if stdout else stderr
        return stdout if stdout.strip() else '[WARN] No output. Use print() to see results.'

    def close(self):
        if self._client: self._client.stop_channels()
        if self._owns_kernel and self._km is not None:
            self._km.shutdown_kernel(now=True)
            self._km.cleanup_resources()

    def reset(self):
        # Updated reset to match the new library initializations
        self.execute(
            '%reset -f\nimport sys\nsys.set_int_max_str_digits(0)\nsys.setrecursionlimit(20000)\n'
            'import math\nimport numpy\nimport sympy\nimport scipy\nimport scipy.special\n'
            'import itertools\nimport collections\nimport mpmath\nmpmath.mp.dps = 64\n'
        )

    def __del__(self): self.close()

In [ ]:
class AIMO3Tool:
    def __init__(self, local_jupyter_timeout: float, tool_prompt: str, sandbox=None):
        self._local_jupyter_timeout = local_jupyter_timeout
        self._tool_prompt = tool_prompt
        self._jupyter_session = sandbox
        self._execution_lock = threading.Lock()

    @property
    def tool_config(self) -> ToolNamespaceConfig:
        return ToolNamespaceConfig(name='python', description=self._tool_prompt, tools=[])

    def process_sync_plus(self, message: Message) -> list[Message]:
        with self._execution_lock:
            try: output = self._jupyter_session.execute(message.content[0].text)
            except Exception as exc: output = f'[ERROR] {exc}'
        content = TextContent(text=output)
        msg = Message(author=Author(role=Role.TOOL, name='python'), content=[content]).with_recipient('assistant')
        if message.channel: msg = msg.with_channel(message.channel)
        return [msg]

# ==========================================
# 3. TRACE FORMATTING, PARSING & FILTERING
# ==========================================

def _format_conversation_trace(conversation, attempt_index) -> str:
    """Helper to convert a conversation object into a readable string to save to disk."""
    if not conversation: return f"--- No conversation recorded ---"
    trace_lines = [f"========== TRACE FOR ATTEMPT {attempt_index} =========="]

    for msg in conversation.messages:
        role_str = str(getattr(msg.author, 'role', 'UNKNOWN')).replace('Role.', '').upper()
        author_name = getattr(msg.author, 'name', None)
        channel = getattr(msg, 'channel', None)
        recipient = getattr(msg, 'recipient', None)

        text_parts = []
        if hasattr(msg, 'content') and msg.content:
            for c in (msg.content if isinstance(msg.content, list) else [msg.content]):
                if hasattr(c, 'text'):
                    text_parts.append(str(c.text))
                elif type(c).__name__ == 'SystemContent':
                    sys_text = f"[Identity]: {getattr(c, 'model_identity', '')}\n[Effort]: {getattr(c, 'reasoning_effort', '')}"
                    text_parts.append(sys_text.strip())
                else:
                    text_parts.append(str(c))

        text = "\n".join(text_parts).strip()
        if role_str == 'USER': trace_lines.append(f"\n[🟢 USER]\n{text}")
        elif role_str == 'SYSTEM': trace_lines.append(f"\n[⚙️ SYSTEM]\n{text}")
        elif role_str == 'ASSISTANT':
            if channel == 'analysis': trace_lines.append(f"\n[🧠 ASSISTANT (Channel: analysis) - Thinking]\n{text}")
            elif channel == 'commentary': trace_lines.append(f"\n[⚡ ASSISTANT (Channel: commentary) - Calling Tool: to={recipient}]\n{text}" if recipient else f"\n[💬 ASSISTANT (Channel: commentary) - Preamble]\n{text}")
            elif channel == 'final': trace_lines.append(f"\n[🎯 ASSISTANT (Channel: final) - Final Answer]\n{text}")
            else: trace_lines.append(f"\n[🤖 ASSISTANT]\n{text}")
        elif role_str == 'TOOL' or author_name == 'python':
            trace_lines.append(f"\n[🖥️ TOOL RESPONSE (from={author_name or 'tool'})]\n{text}")
        else: trace_lines.append(f"\n[{role_str}]\n{text}")

    trace_lines.append("\n===========================================")
    return "\n".join(trace_lines)


def scan_for_answer(text: str) -> int | None:
    for pattern in [r'\\boxed\s*\{(?:.*?[=:])?\s*([0-9,]+)\s*\}', r'final\s+answer\s+is\s*([0-9,]+)']:
        matches = re.findall(pattern, text, re.IGNORECASE)
        if matches:
            try:
                val = int(matches[-1].replace(',', ''))
                if 0 <= val <= 99999: return val
            except ValueError: pass
    return None

def extract_problem_and_answer_from_trace(trace_filepath):
    """Extracts problem text and base agent's final answer using exact structure matching."""
    with open(trace_filepath, 'r', encoding='utf-8') as f:
        trace_text = f.read()

    # 1. SAFELY EXTRACT PROBLEM TEXT
    try:
        user_section = trace_text.split('[🟢 USER]\n')[1]

        # Safely split at the next Role marker (avoids breaking on math matrices like \n[1, 2])
        user_section = re.split(r'\n\[(?:🧠|⚡|💬|🎯|🤖|🖥️|⚙️)', user_section, maxsplit=1)[0]

        # Strip out the injected system/tool prompts
        problem_text = user_section.split('You have access to `math`')[0]
        problem_text = problem_text.split('Use this tool to execute')[0]
        problem_text = problem_text.strip()
    except IndexError:
        problem_text = "ERROR: Could not parse problem text."

    # 2. SAFELY EXTRACT ANSWER (IGNORE PROMPT INSTRUCTIONS)
    # Split the trace at the first ASSISTANT marker.
    # This guarantees we only scan what the model ACTUALLY GENERATED.
    parts = re.split(r'\[(?:🧠|⚡|💬|🎯|🤖) ASSISTANT', trace_text, maxsplit=1)

    if len(parts) > 1:
        text_to_scan = parts[1]
    else:
        # Fallback if no assistant output exists (prevents crashing)
        text_to_scan = ""

    answer = scan_for_answer(text_to_scan)

    return problem_text, trace_text, answer



def get_judger_triggered_questions(directories, val_df, require_correct_base=True):
    questions_to_rerun = []

    for folder in directories:
        if not os.path.exists(folder): continue

        all_files = os.listdir(folder)
        judger_q_ids = set([re.match(r'q(\d+)_judger_attempt_\d+\.txt', f).group(1)
                            for f in all_files if re.match(r'q(\d+)_judger_attempt_\d+\.txt', f)])

        for q_id in judger_q_ids:
            base_files = [f for f in all_files if re.match(rf'^q{q_id}_attempt_\d+\.txt$', f)]
            base_files.sort(key=lambda x: int(re.search(r'attempt_(\d+)', x).group(1)))
            base_file_paths = [os.path.join(folder, bf) for bf in base_files]

            if not base_file_paths: continue

            prob_txt, _, _ = extract_problem_and_answer_from_trace(base_file_paths[0])

            search_str = prob_txt[:100]
            matched_rows = val_df[val_df['problem'].str.contains(search_str, regex=False, na=False)]

            if len(matched_rows) == 0: continue
            expected_ans = int(float(matched_rows.iloc[0]['answer']))

            base_answers = []
            for filepath in base_file_paths:
                _, _, ans = extract_problem_and_answer_from_trace(filepath)
                base_answers.append(ans)

            has_correct_base = any(ans == expected_ans for ans in base_answers if ans is not None)

            if require_correct_base and not has_correct_base:
                continue

            questions_to_rerun.append({
                'experiment_folder': folder,
                'q_id': q_id,
                'base_files': base_file_paths,
                'base_answers': base_answers,
                'expected_answer': expected_ans
            })

    return questions_to_rerun


In [ ]:
class OfflineJudgerSolver:
    def __init__(self, cfg, port: int = 8000):
        self.cfg = cfg
        self.port = port
        self.base_url = f'http://0.0.0.0:{port}/v1'
        self.api_key = 'sk-local'
        self.template = AIMO3Template()
        self.encoding = load_harmony_encoding(HarmonyEncodingName.HARMONY_GPT_OSS)
        self.stop_token_ids = self.encoding.stop_tokens_for_assistant_actions()

        if self.cfg.save_traces:
            os.makedirs(self.cfg.trace_dir, exist_ok=True)

        self._preload_model_weights()
        self.server_process = self._start_server()
        self.client = OpenAI(base_url=self.base_url, api_key=self.api_key, timeout=self.cfg.session_timeout)

        self._wait_for_server()
        self._initialize_kernels()

    def _preload_model_weights(self) -> None:
        print(f'Loading model weights into RAM...')
        files_to_load = []
        for path in [self.cfg.model_path, self.cfg.draft_model_path]:
            if not path or not os.path.exists(path): continue
            for root, _, files in os.walk(path):
                for f in files: files_to_load.append(os.path.join(root, f))
        def _read_file(p):
            with open(p, 'rb') as f:
                while f.read(1024 * 1024 * 1024): pass
        with ThreadPoolExecutor(max_workers=self.cfg.workers) as executor:
            list(executor.map(_read_file, files_to_load))

    def _start_server(self) -> subprocess.Popen:
        spec_config_json = f'{{"model":"{self.cfg.draft_model_path}","num_speculative_tokens":3,"method":"eagle3","draft_tensor_parallel_size":1}}'
        cmd =[
            sys.executable, '-m', 'vllm.entrypoints.openai.api_server',
            '--seed', str(self.cfg.seed), '--model', self.cfg.model_path,
            '--served-model-name', self.cfg.served_model_name, '--tensor-parallel-size', '1',
            '--max-num-seqs', str(self.cfg.batch_size), '--gpu-memory-utilization', str(self.cfg.gpu_memory_utilization),
            '--host', '0.0.0.0', '--port', str(self.port), '--dtype', self.cfg.dtype,
            '--kv-cache-dtype', self.cfg.kv_cache_dtype, '--max-model-len', str(self.cfg.context_tokens),
            '--stream-interval', str(self.cfg.stream_interval), '--async-scheduling', '--enable-prefix-caching',
            '--speculative-config', spec_config_json, '--swap-space', '16'
        ]
        self.log_file = open('vllm_server.log', 'w')
        return subprocess.Popen(cmd, stdout=self.log_file, stderr=subprocess.STDOUT, start_new_session=True)

    def _wait_for_server(self):
        print('Waiting for vLLM server...')
        start_time = time.time()
        for _ in range(self.cfg.server_timeout):
            if self.server_process.poll() is not None:
                raise RuntimeError('Server died.')
            try:
                self.client.models.list()
                print(f'Server ready in {time.time() - start_time:.2f}s.\n')
                return
            except Exception: time.sleep(1)
        raise RuntimeError('Server start timeout.')

    def _initialize_kernels(self) -> None:
        self.sandbox_pool = queue.Queue()
        with ThreadPoolExecutor(max_workers=self.cfg.workers) as executor:
            futures = [executor.submit(lambda: AIMO3Sandbox(timeout=self.cfg.jupyter_timeout)) for _ in range(self.cfg.workers)]
            for f in as_completed(futures): self.sandbox_pool.put(f.result())

    def _generate_summary(self, problem: str, trace_text: str, q_id: str, attempt_index: int, attempt_seed: int, deadline: float) -> dict:

        start_time = time.time()
        input_tokens = 0
        output_tokens = 0
        summary_text = ""

        summarizer_user_prompt = (
            f"Here is the original problem:\n{problem}\n\n"
            f"Here is the raw trace of the solver's thought process and code executions:\n"
            f"{trace_text}\n\n"
            f"Based ONLY on the trace above, write a clean, comprehensive and objective summary of the solution. "
            f"Keep it lean but detailed enough for a complete verification by the judger.Judger will compare it with other provided solution summaries"
        )

        sys_content = SystemContent.new().with_model_identity(self.cfg.summarizer_system_prompt).with_reasoning_effort(self.cfg.summarizer_reasoning_effort)
        messages = [Message.from_role_and_content(Role.SYSTEM, sys_content), Message.from_role_and_content(Role.USER, summarizer_user_prompt)]
        conversation = Conversation.from_messages(messages)

        try:
            prompt_ids = self.encoding.render_conversation_for_completion(conversation, Role.ASSISTANT)
            input_tokens = len(prompt_ids)
            max_tokens = min(10000, self.cfg.context_tokens - input_tokens)

            if max_tokens > 100:
                stream = self.client.completions.create(
                    model=self.cfg.served_model_name, max_tokens=max_tokens, prompt=prompt_ids,
                    seed=attempt_seed, stream=True, extra_body={'stop_token_ids': self.stop_token_ids, 'return_token_ids': True}
                )
                sum_token_buffer = []
                for chunk in stream:
                    if time.time() > deadline: break
                    choice = chunk.choices[0]
                    t_ids = getattr(choice, 'token_ids', None) or (choice.model_extra.get('token_ids', []) if hasattr(choice, 'model_extra') and choice.model_extra else [])
                    if t_ids: sum_token_buffer.extend(t_ids)
                stream.close()

                output_tokens = len(sum_token_buffer)

                if sum_token_buffer:
                    sum_msgs = self.encoding.parse_messages_from_completion_tokens(sum_token_buffer, Role.ASSISTANT)
                    conversation.messages.extend(sum_msgs)

                    # Correctly filter to ONLY include 'final' channel / standard text content
                    parts = [
                        str(c.text) for m in sum_msgs
                        if getattr(m, 'channel', None) in ('final', None)
                        for c in m.content if hasattr(c, 'text')
                    ]
                    summary_text = "\n".join(parts).strip()

        except Exception as e:
            print(f"Summary Error: {e}")

        duration = time.time() - start_time
        is_valid = bool(summary_text.strip())

        # SAVE SUMMARIZER TRACE
        if getattr(self.cfg, 'save_traces', False):
            trace_log = _format_conversation_trace(conversation, attempt_index=f"OFFLINE_SUMMARIZER_{attempt_index}")
            with open(f"{self.cfg.trace_dir}/q{q_id}_offline_summary_attempt_{attempt_index}.txt", 'w', encoding='utf-8') as f:
                f.write(trace_log)

        return {
            'Summary': summary_text,
            'Summ_In_Tokens': input_tokens,
            'Summ_Out_Tokens': output_tokens,
            'Summ_Time': duration,
            'Summ_Valid': is_valid
        }

    def _process_judger_attempt(self, judger_prompt: str, q_id: str, node_index: int, attempt_seed: int, deadline: float) -> dict:
        start_time = time.time()
        input_tokens = 0
        output_tokens = 0
        python_calls = 0
        python_errors = 0
        first_turn = True
        final_answer = None

        sandbox = self.sandbox_pool.get(timeout=self.cfg.sandbox_timeout)
        local_tool = AIMO3Tool(local_jupyter_timeout=self.cfg.jupyter_timeout, tool_prompt=self.cfg.tool_prompt, sandbox=sandbox)

        tool_config = ToolNamespaceConfig(name='python', description=self.cfg.tool_prompt, tools=[])
        messages = self.template.apply_chat_template(self.cfg.system_prompt, judger_prompt, tool_config)
        conversation = Conversation.from_messages(messages)

        try:
            for _ in range(self.cfg.turns):
                if time.time() > deadline: break

                prompt_ids = self.encoding.render_conversation_for_completion(conversation, Role.ASSISTANT)
                if first_turn:
                    input_tokens = len(prompt_ids)
                    first_turn = False

                max_tokens = self.cfg.context_tokens - len(prompt_ids)
                if max_tokens < self.cfg.buffer_tokens: break

                stream = self.client.completions.create(
                    model=self.cfg.served_model_name, max_tokens=max_tokens, prompt=prompt_ids,
                    seed=attempt_seed, stream=True, extra_body={'stop_token_ids': self.stop_token_ids, 'return_token_ids': True}
                )

                token_buffer, text_chunks = [], []
                for chunk in stream:
                    if time.time() > deadline: break
                    choice = chunk.choices[0]
                    new_tokens = getattr(choice, 'token_ids', None) or (choice.model_extra.get('token_ids', []) if hasattr(choice, 'model_extra') and choice.model_extra else [])
                    if new_tokens:
                        token_buffer.extend(new_tokens)
                        text_chunks.append(choice.text)
                    if '}' in choice.text:
                        ans = scan_for_answer(''.join(text_chunks[-self.cfg.search_tokens:]))
                        if ans is not None and ans != 42:
                            final_answer = ans
                            break
                stream.close()
                output_tokens += len(token_buffer)

                if final_answer is not None:
                    if token_buffer:
                        try: conversation.messages.extend(self.encoding.parse_messages_from_completion_tokens(token_buffer, Role.ASSISTANT))
                        except Exception: pass
                    break

                if not token_buffer: break

                new_messages = self.encoding.parse_messages_from_completion_tokens(token_buffer, Role.ASSISTANT)
                conversation.messages.extend(new_messages)
                last_message = new_messages[-1]
                message_text = "".join([str(c.text) for c in last_message.content if hasattr(c, 'text')])

                if last_message.channel == 'final':
                    final_answer = scan_for_answer(message_text)
                    break

                if last_message.recipient == 'python':
                    python_calls += 1
                    tool_responses = local_tool.process_sync_plus(last_message)
                    resp_text = tool_responses[0].content[0].text
                    if resp_text.startswith('[ERROR]') or 'Traceback' in resp_text or 'Error:' in resp_text:
                        python_errors += 1
                    conversation.messages.extend(tool_responses)
                else:
                    ans = scan_for_answer(message_text)
                    if ans is not None:
                        final_answer = ans
                        break
        except Exception as e:
            python_errors += 1
        finally:
            sandbox.reset()
            self.sandbox_pool.put(sandbox)

            # SAVE JUDGER TRACE
            if getattr(self.cfg, 'save_traces', False):
                trace_log = _format_conversation_trace(conversation, attempt_index=f"OFFLINE_JUDGER_NODE_{node_index}")
                with open(f"{self.cfg.trace_dir}/q{q_id}_offline_judger_node_{node_index}.txt", 'w', encoding='utf-8') as f:
                    f.write(trace_log)

        execution_time = time.time() - start_time
        return {
            'Attempt': node_index,
            'Answer': final_answer,
            'In Tokens': input_tokens,
            'Out Tokens': output_tokens,
            'Time (s)': execution_time,
            'Python Calls': python_calls,
            'Python Errors': python_errors
        }

    def _decide_final_answer(self, base_results: list, judger_results: list) -> int:
        candidates = set()
        judger_counts = Counter()
        total_counts = Counter()
        token_sums = defaultdict(int)
        token_counts = defaultdict(int)

        for res in judger_results:
            ans = res.get('Answer')
            if ans is not None:
                candidates.add(ans)
                judger_counts[ans] += 1
                total_counts[ans] += 1
                token_sums[ans] += res.get('Out Tokens', 0)
                token_counts[ans] += 1

        for res in base_results:
            ans = res.get('Answer')
            if ans is not None:
                candidates.add(ans)
                total_counts[ans] += 1

                # Approximate Base Agent Output Tokens accurately by ignoring USER and TOOL traces
                trace = res.get('Trace', '')
                assistant_texts = re.findall(r'\[(?:🧠|⚡|💬|🎯|🤖) ASSISTANT.*?\]\n(.*?)(?=\n\[|$)', trace, re.DOTALL)
                assistant_only_len = sum(len(text) for text in assistant_texts)

                token_sums[ans] += assistant_only_len // 4
                token_counts[ans] += 1

        if not candidates: return 0

        scoring = []
        for ans in candidates:
            avg_tokens = token_sums[ans] / max(1, token_counts[ans])
            scoring.append({
                'Answer': ans,
                'Judger Votes': judger_counts[ans],
                'Total Votes': total_counts[ans],
                'Avg Tokens': avg_tokens
            })

        scoring.sort(key=lambda x: (x['Judger Votes'], x['Total Votes'], -x['Avg Tokens']), reverse=True)
        return scoring[0]['Answer']

    def evaluate_offline(self, q_id, base_trace_paths, expected_answer):
        print(f"\n{'='*70}\n🔄 OFFLINE EVALUATION: Question {q_id} (Expected Answer: {expected_answer})\n{'='*70}")
        detailed_results = []
        problem_text = None

        # 1. Parse base traces
        for i, filepath in enumerate(base_trace_paths):
            attempt_idx = i + 1
            prob_txt, full_trace, answer = extract_problem_and_answer_from_trace(filepath)
            if problem_text is None: problem_text = prob_txt

            attempt_seed = int(math.pow(self.cfg.seed + attempt_idx - 1, 2))
            detailed_results.append({'Attempt': attempt_idx, 'Trace': full_trace, 'Answer': answer, 'Seed': attempt_seed})

        # 2. Run Summarizer concurrently
        print("\n📝 Running Summarizers...")
        judger_deadline = time.time() + getattr(self.cfg, 'judger_timeout', 600)
        summarizer_futures = {}

        with ThreadPoolExecutor(max_workers=self.cfg.workers) as sum_executor:
            for res in detailed_results:
                # ONLY SUMMARIZE IF THE ATTEMPT PRODUCED A VALID ANSWER
                if res['Answer'] is not None:
                    summarizer_futures[sum_executor.submit(
                        self._generate_summary, problem_text, res['Trace'], q_id, res['Attempt'], res['Seed'], judger_deadline
                    )] = res

            sum_metrics = []
            for future in as_completed(summarizer_futures):
                res = summarizer_futures[future]
                try:
                    summ_info = future.result()
                    res['Summary'] = summ_info['Summary']
                    sum_metrics.append({
                        'Attempt': res['Attempt'],
                        'Answer': res['Answer'],
                        'In Tokens': summ_info['Summ_In_Tokens'],
                        'Out Tokens': summ_info['Summ_Out_Tokens'],
                        'Time (s)': round(summ_info['Summ_Time'], 2),
                        'Valid': summ_info['Summ_Valid']
                    })
                except Exception as e:
                    print(f"Summarizer Error for attempt {res['Attempt']}: {e}")

        # PRINT Kaggle-style Summarizer Metrics
        if sum_metrics:
            print("\n--- Summarizer Node Metrics ---")
            display(pd.DataFrame(sum_metrics))

        # 3. Build Kaggle-Style Judger Prompt
        summaries_text = ""
        valid_results = [r for r in detailed_results if r['Answer'] is not None and r.get('Summary')]
        for res in valid_results:
            summaries_text += f"### Candidate from Attempt {res['Attempt']} (Proposed Answer: {res['Answer']}) ###\n{res['Summary']}\n\n"

        judger_detailed_results = []

        if summaries_text.strip():
            print(f"\n⚖️ Triggering {self.cfg.judger_attempts} Judger nodes...")
            judger_prompt = (
                f"You are an expert mathematical judge. Below is a math problem, followed by several candidate "
                f"solution summaries from different ai solvers.\n\n"
                f"Problem:\n{problem_text}\n\n"
                f"{self.cfg.preference_prompt}\n\n"
                f"Summaries:\n{summaries_text}\n"
                f"Your task:\n"
                f"1. Carefully read the question understand it's intend and evaluate the mathematical reasoning in each candidate summary.\n"
                f"3. Write Python code when nedeed to explicitly test divergent claims presented in summaries to prove which one is correct.\n"
                f"4. Provide your final correct answer inside \\boxed{{}} (e.g. \\boxed{{42}}).\n\n"
                f"Ensure your final answer is a non-negative integer."
            )

            # 4. Run Judger
            with ThreadPoolExecutor(max_workers=self.cfg.workers) as j_exec:
                j_futures = [j_exec.submit(self._process_judger_attempt, judger_prompt, q_id, i+1, self.cfg.seed+1000+i, judger_deadline) for i in range(self.cfg.judger_attempts)]
                for f in as_completed(j_futures):
                    try: judger_detailed_results.append(f.result())
                    except Exception: pass

            # PRINT Kaggle-style Judger Metrics
            if judger_detailed_results:
                print("\n--- Judger Node Metrics ---")
                display(pd.DataFrame(judger_detailed_results).drop(columns=['Trace'], errors='ignore'))

        final_answer = self._decide_final_answer(detailed_results, judger_detailed_results)
        print(f"\nSUBMITTED ANSWER: {final_answer}")
        print(f"EXPECTED ANSWER : {expected_answer}")
        print(f"JUDGER CORRECT? : {'✅ YES' if str(final_answer) == str(expected_answer) else '❌ NO'}")

        return final_answer

    def __del__(self):
        if hasattr(self, 'server_process'):
            self.server_process.terminate()
            self.server_process.wait()


In [ ]:
# ==========================================
# 5A. TARGET SELECTION (CELL 1)
# ==========================================

import pandas as pd

# 1. Make sure you have loaded your dataframe!
df = pd.read_csv("/content/aimo3_val.csv")

# 2. Directories containing your saved traces
folders_to_scan = [
    '/content/drive/MyDrive/white_wash_335_atempt_3_judger_speculative_decoding',
    '/content/drive/MyDrive/white_wash_3_atempt_3_judger_speculative_decoding',
    '/content/drive/MyDrive/whitewash_3_atempt_1_judger_speculative_decoding',
    '/content/drive/MyDrive/whitewash_kaggle_match2_all_val_533_new_parser'
    # Add other trace folders here
]

# 3. Toggle: Only evaluate traces where AT LEAST ONE base answer was correct
REQUIRE_CORRECT_BASE = True

print("🔍 Scanning folders for target questions...")
raw_offline_targets = get_judger_triggered_questions(
    directories=folders_to_scan,
    val_df=df,
    require_correct_base=REQUIRE_CORRECT_BASE
)

# ---------------------------------------------------------
# 🛑 NEW: FILTER OUT ALL 42s (EXPECTED OR PREDICTED)
# ---------------------------------------------------------
offline_targets = []
for target in raw_offline_targets:
    # Skip if the actual expected answer is 42
    if target['expected_answer'] == 42:
        continue
    # Skip if any of the base attempts answered 42 (hallucinations from prompt)
    if 42 in target['base_answers']:
        continue

    # If it passes the checks, keep it
    offline_targets.append(target)

removed_count = len(raw_offline_targets) - len(offline_targets)
if removed_count > 0:
    print(f"🚫 Filtered out {removed_count} questions containing '42' as an expected or predicted answer.")
# ---------------------------------------------------------


# --- PRE-RUN REPORTING ---
print(f"\n{'='*60}")
print(f"🎯 TARGET SELECTION SUMMARY")
print(f"{'='*60}")
if len(offline_targets) == 0:
    print("No valid questions found matching criteria.")
else:
    target_summary = []
    for t in offline_targets:
        target_summary.append({
            'Folder': t['experiment_folder'].split('/')[-1][:20] + '...',
            'Q_ID': t['q_id'],
            'Expected Ans': t['expected_answer'],
            'Base Answers Proposed': str(t['base_answers'])
        })
    display(pd.DataFrame(target_summary))
    print(f"\n✅ {len(offline_targets)} Targets successfully selected and saved to memory.")
    print(">>> Proceed to the NEXT CELL to initialize the server and run the offline evaluation.")

🔍 Scanning folders for target questions...
🚫 Filtered out 4 questions containing '42' as an expected or predicted answer.

🎯 TARGET SELECTION SUMMARY


,Folder,Q_ID,Expected Ans,Base Answers Proposed
0,white_wash_335_atemp...,42,8700,"[8700, 8283, 7450, 8250, None]"
1,white_wash_335_atemp...,73,59,"[155, 0, 47, 215, 59]"
2,white_wash_335_atemp...,78,93,"[93, 148, 93, 144, 144]"
3,white_wash_335_atemp...,5,506,"[8, 506, 1008, 1008, 6]"
4,white_wash_335_atemp...,91,125,"[125, 3136, 3136, 125, 284]"
5,white_wash_335_atemp...,9,22,"[22, 8, 8, 24, 22]"
6,white_wash_335_atemp...,53,46,"[440, 516, 516, 46, 46]"
7,white_wash_335_atemp...,85,2079,"[2079, 0, 1055, None, 2079]"
8,white_wash_3_atempt_...,47,70,"[63, 70, 62]"
9,white_wash_3_atempt_...,88,48,"[48, 40, 48]"



✅ 45 Targets successfully selected and saved to memory.
>>> Proceed to the NEXT CELL to initialize the server and run the offline evaluation.


In [ ]:
import pandas as pd

# 1. Define the exact Q_IDs you want to isolate (ensure they are strings to match regex extraction)
target_q_ids = ['7', '65', '78', '100']

# 2. Filter the existing offline_targets list
sanitized_targets = [t for t in offline_targets if str(t['q_id']) in target_q_ids]

# 3. Overwrite the main list so CELL 2 picks up the sanitized version
offline_targets = sanitized_targets

# 4. Display the new sanitized summary
print(f"\n{'='*60}")
print(f"🧹 SANITIZED TARGET SELECTION SUMMARY")
print(f"{'='*60}")

if len(offline_targets) == 0:
    print("⚠️ No targets found matching those specific IDs in the current list.")
else:
    target_summary = []
    for t in offline_targets:
        target_summary.append({
            'Folder': t['experiment_folder'].split('/')[-1][:20] + '...',
            'Q_ID': t['q_id'],
            'Expected Ans': t['expected_answer'],
            'Base Answers Proposed': str(t['base_answers'])
        })
    display(pd.DataFrame(target_summary))
    print(f"\n✅ Successfully sanitized down to {len(offline_targets)} targets.")
    print(">>> You can now proceed to CELL 2 to initialize the server and run the offline evaluation.")


🧹 SANITIZED TARGET SELECTION SUMMARY


,Folder,Q_ID,Expected Ans,Base Answers Proposed
0,white_wash_335_atemp...,78,93,"[93, 148, 93, 144, 144]"
1,white_wash_3_atempt_...,78,93,"[144, 93, 93]"
2,white_wash_3_atempt_...,65,1,"[None, 1, 2]"
3,white_wash_3_atempt_...,7,140,"[100, 140, 140]"
4,whitewash_3_atempt_1...,100,97665,"[None, 255, 97665]"
5,whitewash_3_atempt_1...,78,93,"[52, 148, 93]"
6,whitewash_3_atempt_1...,7,140,"[140, 100, 140]"



✅ Successfully sanitized down to 7 targets.
>>> You can now proceed to CELL 2 to initialize the server and run the offline evaluation.


In [ ]:
# ==========================================
# 5B. OFFLINE INFERENCE (CELL 2)
# ==========================================

if 'offline_targets' not in globals() or len(offline_targets) == 0:
    print("⚠️ No targets found in memory! Please run the Target Selection cell above first.")
else:
    print("\n🚀 Initializing Offline Solver & vLLM Server...")
    # Initialize the solver (loads model weights & starts vLLM)
    offline_solver = OfflineJudgerSolver(CFG)

    print(f"\n{'='*50}\nStarting offline experiments on {len(offline_targets)} highly-curated questions...\n{'='*50}")

    offline_results = []
    for target in offline_targets:

        # ---> FIX: Create a unique identifier to prevent overwriting trace files <---
        folder_name = target['experiment_folder'].split('/')[-1]
        unique_q_id = f"{target['q_id']}_{folder_name}"

        predicted_answer = offline_solver.evaluate_offline(
            q_id=unique_q_id,  # <-- Pass the combined ID here
            base_trace_paths=target['base_files'],
            expected_answer=target['expected_answer']
        )

        offline_results.append({
            'Folder': folder_name,
            'Q_ID': target['q_id'],
            'Expected': target['expected_answer'],
            'Judger_Predicted': predicted_answer,
            'Judger_Fixed_It?': str(predicted_answer) == str(target['expected_answer'])
        })

    print("\n🎉 All Offline Tests Complete!")
    print(f"📁 New traces saved to: {CFG.trace_dir}")

    # ==========================================
    # Final Reporting & Accuracy Tables
    # ==========================================
    results_df = pd.DataFrame(offline_results)

    print("\n" + "="*70)
    print("📋 DETAILED PREDICTIONS")
    print("="*70)
    display(results_df)

    if len(results_df) > 0:
        total_evals = len(results_df)
        total_success = results_df['Judger_Fixed_It?'].sum()
        overall_acc = (total_success / total_evals) * 100

        print("\n" + "="*70)
        print("🏆 OVERALL OFFLINE EVALUATION RESULTS")
        print("="*70)
        print(f"Total Questions Evaluated : {total_evals}")
        print(f"Successfully Fixed        : {total_success}")
        print(f"Overall Success Rate      : {overall_acc:.2f}%\n")

        print("📊 ACCURACY BREAKDOWN BY EXPERIMENT FOLDER:")
        # Group by Folder to see which base traces performed best with the current Judger
        summary_df = results_df.groupby('Folder').agg(
            Total_Evaluated=('Q_ID', 'count'),
            Successfully_Fixed=('Judger_Fixed_It?', 'sum')
        ).reset_index()

        # Calculate percentage and sort by highest accuracy
        summary_df['Accuracy (%)'] = (summary_df['Successfully_Fixed'] / summary_df['Total_Evaluated'] * 100).round(2)
        summary_df = summary_df.sort_values('Accuracy (%)', ascending=False).reset_index(drop=True)

        display(summary_df)


🚀 Initializing Offline Solver & vLLM Server...
Loading model weights into RAM...
Waiting for vLLM server...
Server ready in 120.20s.


Starting offline experiments on 7 highly-curated questions...

🔄 OFFLINE EVALUATION: Question 78_white_wash_335_atempt_3_judger_speculative_decoding (Expected Answer: 93)

📝 Running Summarizers...

--- Summarizer Node Metrics ---


,Attempt,Answer,In Tokens,Out Tokens,Time (s),Valid
0,1,93,35556,1717,56.70,True
1,4,144,46749,2118,66.94,True
2,2,148,40560,2471,70.04,True
3,3,93,28167,2471,70.51,True
4,5,144,32781,2488,70.82,True



⚖️ Triggering 3 Judger nodes...

--- Judger Node Metrics ---


,Attempt,Answer,In Tokens,Out Tokens,Time (s),Python Calls,Python Errors
0,2,93,9641,11061,162.053411,20,0
1,3,93,9641,12387,177.532257,8,0
2,1,93,9641,13641,187.292620,15,0



SUBMITTED ANSWER: 93
EXPECTED ANSWER : 93
JUDGER CORRECT? : ✅ YES

🔄 OFFLINE EVALUATION: Question 78_white_wash_3_atempt_3_judger_speculative_decoding (Expected Answer: 93)

📝 Running Summarizers...

--- Summarizer Node Metrics ---


...

,Attempt,Answer,In Tokens,Out Tokens,Time (s),Python Calls,Python Errors
0,1,140,6102,6448,71.824910,11,0
1,3,140,6102,10720,134.345844,18,2
2,2,140,6102,14517,172.816267,28,2



SUBMITTED ANSWER: 140
EXPECTED ANSWER : 140
JUDGER CORRECT? : ✅ YES

🎉 All Offline Tests Complete!
📁 New traces saved to: /content/traces

📋 DETAILED PREDICTIONS


,Folder,Q_ID,Expected,Judger_Predicted,Judger_Fixed_It?
0,white_wash_335_atempt_3_judger_speculative_dec...,78,93,93,True
1,white_wash_3_atempt_3_judger_speculative_decoding,78,93,93,True
2,white_wash_3_atempt_3_judger_speculative_decoding,65,1,1,True
3,white_wash_3_atempt_3_judger_speculative_decoding,7,140,140,True
4,whitewash_3_atempt_1_judger_speculative_decoding,100,97665,97665,True
5,whitewash_3_atempt_1_judger_speculative_decoding,78,93,52,False
6,whitewash_3_atempt_1_judger_speculative_decoding,7,140,140,True



🏆 OVERALL OFFLINE EVALUATION RESULTS
Total Questions Evaluated : 7
Successfully Fixed        : 6
Overall Success Rate      : 85.71%

📊 ACCURACY BREAKDOWN BY EXPERIMENT FOLDER:


,Folder,Total_Evaluated,Successfully_Fixed,Accuracy (%)
0,white_wash_335_atempt_3_judger_speculative_dec...,1,1,100.00
1,white_wash_3_atempt_3_judger_speculative_decoding,3,3,100.00
2,whitewash_3_atempt_1_judger_speculative_decoding,3,2,66.67


In [ ]:
!zip -r Prompt1_7_questions.zip /content/traces

  adding: content/traces/ (stored 0%)
  adding: content/traces/q78_white_wash_335_atempt_3_judger_speculative_decoding_offline_summary_attempt_4.txt (deflated 71%)
  adding: content/traces/q78_white_wash_335_atempt_3_judger_speculative_decoding_offline_judger_node_2.txt (deflated 69%)
  adding: content/traces/q7_white_wash_3_atempt_3_judger_speculative_decoding_offline_summary_attempt_2.txt (deflated 70%)
  adding: content/traces/q7_white_wash_3_atempt_3_judger_speculative_decoding_offline_judger_node_2.txt (deflated 69%)
  adding: content/traces/q100_whitewash_3_atempt_1_judger_speculative_decoding_offline_summary_attempt_2.txt (deflated 70%)
  adding: content/traces/q78_white_wash_335_atempt_3_judger_speculative_decoding_offline_summary_attempt_3.txt (deflated 68%)
  adding: content/traces/q78_white_wash_3_atempt_3_judger_speculative_decoding_offline_judger_node_1.txt (deflated 68%)
  adding: content/traces/q7_whitewash_3_atempt_1_judger_speculative_decoding_offline_judger_node_1.txt

In [ ]:
!cp -r /content/traces /content/drive/MyDrive/Prompt1_7_questions